In [8]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                           roc_auc_score, roc_curve, auc,
                           precision_recall_curve, average_precision_score)
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

In [2]:
# AdaBoost 실습: 유방암 진단

In [5]:
# 1. 유방암 데이터 로드 (결측치 없음, 이진 분류)
data = load_breast_cancer()
X, y = data.data, data.target

feature_names = data.feature_names
target_names = data.target_names

In [6]:
# 2. 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [8]:
# 3. AdaBoost 모델 생성
ab = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),  # Decision Stump 
    # depth=1 약한 분류기로 만들기 위해서
    n_estimators=50, 
    algorithm='SAMME',    
    random_state=42) 
# learning_rate 학습률 : 학습률에 따라 가중치가 달라짐  수업에서는 기본값으로 둠

In [19]:
ab.fit(X_train, y_train)

/opt/anaconda3/envs/myenv/lib/python3.12/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(


,estimator,DecisionTreeC...r(max_depth=1)
,n_estimators,50
,learning_rate,1.0
,algorithm,'SAMME'
,random_state,42
,criterion,'gini'
,splitter,'best'
,max_depth,1
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0


In [20]:
ab.score(X_test, y_test)

0.9590643274853801

In [28]:
# 5. 예측
y_pred = ab.predict(X_test)
y_proba = ab.predict_proba(X_test)[:, 1]  # y가 2차원이라 1차원으로 바꿔줌

In [23]:
# Confusion Matrix
confusion_matrix(y_test, y_pred)

array([[ 58,   6],
       [  1, 106]])

In [25]:
print(classification_report(y_test, y_pred, target_names=target_names))

              precision    recall  f1-score   support

   malignant       0.98      0.91      0.94        64
      benign       0.95      0.99      0.97       107

    accuracy                           0.96       171
   macro avg       0.96      0.95      0.96       171
weighted avg       0.96      0.96      0.96       171



In [29]:
# ROC-AUC Score
roc_score = roc_auc_score(y_test, y_proba)
print(f"\nROC-AUC Score: {roc_score:.4f}")


ROC-AUC Score: 0.9909


# DMatrix 

In [30]:
!pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 30.6 MB/s  0:00:00


In [1]:
import xgboost as xgb

In [9]:
# DMatrix 생성 (XGBoost 전용 데이터 구조 - 메모리 효율적)
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

# 파라미터 설정
params = {
    'objective': 'binary:logistic',
    'max_depth': 3,
    'learning_rate': 0.1,
    'eval_metric': 'logloss'}

# 학습 (조기 종료 포함)
model = xgb.train(
    params,
    dtrain,
    num_boost_round=100,
    evals=[(dtrain, 'train'), (dtest, 'test')],
    early_stopping_rounds=10,
    verbose_eval=20)

# 예측
y_pred_proba = model.predict(dtest)
y_pred = (y_pred_proba > 0.5).astype(int)
print(f"정확도: {accuracy_score(y_test, y_pred):.4f}")

[0]	train-logloss:0.57736	test-logloss:0.58459
[20]	train-logloss:0.10858	test-logloss:0.17576
[40]	train-logloss:0.04354	test-logloss:0.12087
[60]	train-logloss:0.02366	test-logloss:0.11004
[80]	train-logloss:0.01564	test-logloss:0.10180
[99]	train-logloss:0.01248	test-logloss:0.10146
정확도: 0.9532


# AdaBoost 튜닝 예제

In [10]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

In [13]:
# 파라미터 그리드
param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.5, 1.0, 1.5],
    'estimator': [
        DecisionTreeClassifier(max_depth=1),
        DecisionTreeClassifier(max_depth=2)]}

# Grid Search
ada_grid = GridSearchCV(
    AdaBoostClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1)

ada_grid.fit(X_train, y_train)
print("최적 파라미터:", ada_grid.best_params_)
print(f"최적 CV 점수: {ada_grid.best_score_:.4f}")
print(f"테스트 정확도: {ada_grid.score(X_test, y_test):.4f}")

최적 파라미터: {'estimator': DecisionTreeClassifier(max_depth=2), 'learning_rate': 1.0, 'n_estimators': 50}
최적 CV 점수: 0.9750
테스트 정확도: 0.9649
